In [1]:
%pip install -q torch nvidia-ml-py3 librosa numpy

# Using CUDA 13.0
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu130
# pip install transformers

import os
import gc
import json
import numpy as np
import pickle
import gzip
import math
import librosa # For audio processing
from pathlib import Path
from typing import Tuple, List, Dict
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, Subset, DataLoader, WeightedRandomSampler, random_split
from torch.optim.lr_scheduler import LinearLR, SequentialLR
from torch.amp import GradScaler, autocast
from transformers import ASTModel, ASTConfig, ASTFeatureExtractor

import pynvml
import psutil
from tqdm import tqdm
from sklearn.metrics import f1_score

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:512'

class Config:
    # Paths
    PARENT_DIR = os.path.dirname(os.getcwd())
    DATA_DIR = PARENT_DIR + "/Data"
    PREPROCESSED_DIR = PARENT_DIR + "/Preprocessed"
    OUTPUT_DIR = PARENT_DIR + "/Model_Output"

    # Preprocessor parameters
    AUDIO_EPOCH_DURATION=10 # 10s clips preferred for AST, 30s causes memory errors
    AUDIO_SAMPLE_RATE=16000
    NUM_AUDIO_FILES=50
    OVERLAP_THRESHOLD = 0.5
    MIN_AUDIO_QUALITY = 0.3

    # Feature parameters
    USE_PREEXTRACTED_FEATURES = True
    
    # Model parameters
    CONTEXT_EPOCHS = 20 # Study was 14 Context -> 10 Output [Had an error when trying to up]
    OUTPUT_EPOCHS = 10 # Study was 14 Context -> 10 Output
    NUM_CLASSES = 3
    DROPOUT = 0.4

    # Architectures
    USE_LSTM = True
    LSTM_LAYERS = 2
    USE_TRANSFORMER_ENCODER = True
    TRANSFORMER_LAYERS = 4
    
    # Training parameters
    BATCH_SIZE = 64
    GRADIENT_ACCUMULATION_STEPS = 8 # Effective batch = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
    NUM_EPOCHS = 50

    LEARNING_RATE_WARMUP_EPOCHS = 5
    LEARNING_RATE = 8e-5
    LEARNING_RATE_MIN = 5e-7
    
    PATIENCE = 10
    SCHEDULER_PATIENCE = 5 # OLD: Not needed with warmup+cosine
    
    OPTIMIZER_WEIGHT_DECAY = 0.6

    FOCAL_GAMMA = 2.0 # Focus on hard examples
    LABEL_SMOOTHING = 0.00 # > 0 might be obscuring minority events
    
    MAX_GRAD_NORM = 1.0 # Gradient clipping norm
    
    # DataLoader parameters
    NUM_WORKERS = 8 # MAX 8
    CACHE_SIZE = 48 # 2 Bad Folders
    USE_COMPRESSION = True
    VAL_SPLIT = 0.2
    
    # Memory optimization
    USE_MIXED_PRECISION = True
    
    # Class weights
    CLASS_WEIGHTS = [1.0, 4.0, 6.0] # Original = [1.0, 1.3, 2.1]
    MINORITY_GRADIENT_SCALE = 2.0

    # Validation
    F1_WEIGHTS = [0.3, 0.35, 0.35] # Importance Split

config = Config()

Note: you may need to restart the kernel to use updated packages.


/etc/python/sitecustomize.py:117: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  mod = _original_import(name, globals, locals, fromlist, level)


In [2]:
class AudioPreprocessor:
    # 30s audio clips at 16kHz by default per study (see below for actual length)
    def __init__(self, data_dir, output_dir, epoch_duration = 30, sample_rate = 16000, use_compression=False, 
                 extract_features=True, overlap_threshold=0.5, min_audio_quality=0.8):
        self.data_dir = Path(data_dir)
        self.output_dir = Path(output_dir)
        self.epoch_duration = epoch_duration
        self.sample_rate = sample_rate
        self.use_compression = use_compression
        self.extract_features = extract_features
        self.overlap_threshold = overlap_threshold
        self.min_audio_quality = min_audio_quality
        
        # AST Input Shape
        self.target_length = sample_rate * epoch_duration

        # Extract Features
        if self.extract_features:
            self.ast_feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
            
            # Load model for feature extraction
            self.ast_model = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
            self.ast_model.eval()
            
            # Move to GPU if available
            if torch.cuda.is_available():
                self.ast_model = self.ast_model.cuda()
                print("    Feature extraction will use GPU")
        
        # Make output directory if it doesn't exist
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        # Indexing
        self.index_path = self.output_dir / "index.json"
        self.folder_metadata = {}
        self.preprocessing_stats = {
            'total_folders': 0,
            'total_epochs': 0,
            'class_distribution': {0: 0, 1: 0, 2: 0},
            'config': {
                'epoch_duration': epoch_duration,
                'sample_rate': sample_rate,
                'overlap_threshold': overlap_threshold,
                'min_audio_quality': min_audio_quality
            }
        }
        
        print(f"\nInitialized AudoPreprocessor")
        print(f"    Data_dir: {data_dir}")
        print(f"    Output_dir: {output_dir}")
        print(f"    Epoch Duration: {epoch_duration}s")
        print(f"    Sample Rate: {sample_rate}Hz")
        print(f"    Samples per epoch: {self.target_length}")
        print(f"    Compression: {'Enabled (gzip)' if use_compression else 'Disabled'}")
        print(f"    Overlap Threshold: {overlap_threshold*100:.0f}%")
        print(f"    Min Audio Quality: {min_audio_quality}")

    def check_audio_quality(self, audio: np.ndarray) -> Dict[str, float]:
        rms_energy = np.sqrt(np.mean(audio**2))
        clipping_ratio = np.sum(np.abs(audio) > 0.999) / len(audio)
        silence_ratio = np.sum(np.abs(audio) < 0.001) / len(audio)

        quality_score = 1.0

        # Completely silent epochs
        if rms_energy < 1e-5:
            quality_score = 0.0

        # Soft penalty for major clipping
        if clipping_ratio > 0.2:
            quality_score *= 0.5

        return {
            'rms_energy': rms_energy,
            'clipping_ratio': clipping_ratio,
            'silence_ratio': silence_ratio,
            'quality_score': quality_score
        }

    def extract_features_batch(self, audio_epochs, apply_normalization=True):
        if not self.extract_features:
            return audio_epochs

        # Batch Processing
        batch_size = 32
        all_features = []

        with torch.no_grad():
            for i in range(0, len(audio_epochs), batch_size):
                batch = audio_epochs[i:i+batch_size]
                
                if apply_normalization:
                    batch = [self.normalize_audio(epoch) for epoch in batch]
                
                # Feature extraction
                inputs = self.ast_feature_extractor(
                    list(batch),
                    sampling_rate=self.sample_rate,
                    return_tensors="pt"
                )
                
                # Move to GPU if available
                if torch.cuda.is_available():
                    inputs = {k: v.cuda() for k, v in inputs.items()}
                
                # Extract features
                outputs = self.ast_model(**inputs)
                features = outputs.last_hidden_state[:, 0, :]  # CLS token
                
                all_features.append(features.cpu().numpy())
        
        return np.vstack(all_features)
    
    def load_annotations(self, folder_id) -> Dict:
        # load annotations from json like 01_annotation.json
        annotation_path = self.data_dir / folder_id / f"{folder_id}_annotation.json"
        
        if (not annotation_path.exists()):
            raise FileNotFoundError(f"Annotation file not found: {annotation_path}")
        
        with open(annotation_path, 'r') as f:
            annotations = json.load(f)
            
        # Debug print statements
        print(f"\nLoaded annotations from {annotation_path}")
        print(f"    Record Start: {annotations['record_start']}s")
        print(f"    Awake Intervals: {len(annotations['awake_intervals'])}")
        print(f"    Events: {len(annotations['events'])}")
        
        return annotations
    
    def normalize_audio(self, audio: np.ndarray, target_db=-20.0) -> np.ndarray:
        # Calculate current RMS
        rms = np.sqrt(np.mean(audio**2))
        
        if rms > 0:
            target_rms = 10 ** (target_db / 20)
            audio = audio * (target_rms / rms)
            audio = np.clip(audio, -1.0, 1.0)
        
        return audio
    
    def load_audio(self, folder_id) -> Tuple[np.ndarray, int]:
        # load audio file like 01_phone.wav
        audio_path = self.data_dir / folder_id / f"{folder_id}_phone.wav"
        
        if (not audio_path.exists()):
            raise FileNotFoundError(f"Audio file not found: {audio_path}")
        
        # Librosa audio loading
        audio, sr = librosa.load(audio_path, sr=self.sample_rate, mono=True)
        
        # Debug print statements
        print(f"\nLoaded audio from {audio_path}")
        print(f"    Audio Shape: {audio.shape}")
        print(f"    Sample Rate: {sr}Hz")
        print(f"    Duration: {len(audio)/sr:.2f}s")
        
        return audio, sr
    
    # Check if time point is within any awake interval
    def is_awake(self, time_point: float, awake_intervals: List[Tuple[float]]) -> bool:
        for start, end in awake_intervals:
            if start <= time_point <= end:
                return True
        return False
        
    # Extract epoch labels based on annotations
    def extract_epoch_labels(self, epoch_start: float, epoch_end: float, events: List[Dict], awake_intervals: List[List[float]]) -> int:
        # Check if epoch during awake interval
        if (self.is_awake(epoch_start, awake_intervals) or self.is_awake(epoch_end, awake_intervals)):
            return -1  # Awake
        
        label = 0 # Default to no event (1 = osa [obstructive sleep apnea], 2 = hyp [hypnopnea])
        
        osa_duration = 0
        hypo_duration = 0
        epoch_duration = epoch_end - epoch_start
        
        # Gonna prioritize in order hypo > osa > none
        for event in events:
            event_start = event['evnet_start']  # Note: typo in original data
            event_end = event_start + event['event_duration']
            event_type = event['event_type']
            
            overlap_start = max(epoch_start, event_start)
            overlap_end = min(epoch_end, event_end)
            
            if overlap_start < overlap_end:  # There is overlap
                overlap_duration = overlap_end - overlap_start
            
                if event_type == 'osa':
                    osa_duration += overlap_duration
                elif event_type == 'hypo':
                    hypo_duration += overlap_duration
            
        threshold = epoch_duration * self.overlap_threshold
        
        if hypo_duration > threshold and hypo_duration > osa_duration:
            label = 2  # Hypopnea is dominant
        elif osa_duration > threshold:
            label = 1  # OSA is dominant
        else:
            label = 0  # No dominant event (mixed or too short)
        
        return label
    
    # Create epochs from audio data and label them
    def create_epochs(self, folder_id: str) -> Tuple[np.ndarray, np.ndarray]:
        # Load data
        annotations = self.load_annotations(folder_id)
        audio, sr = self.load_audio(folder_id)
        
        # Extract Annotations
        record_start = annotations['record_start']
        awake_intervals = annotations['awake_intervals']
        events = annotations['events']
        
        # Number of epochs
        audio_duration = len(audio) / sr
        num_epochs = int(np.floor(audio_duration / self.epoch_duration))
        
        print(f"\nCreating {num_epochs} epochs of {self.epoch_duration}s each from audio of duration {audio_duration:.2f}s")
        
        # Pre-allocate arrays
        epoch_array = []
        label_array = []
        quality_array = []
        
        label_counts = {-2: 0, -1: 0, 0: 0, 1: 0, 2: 0} # -2 low quality
        
        for i in range(num_epochs):
            epoch_start_sample = i * self.target_length
            epoch_end_sample = (i + 1) * self.target_length
            
            # Handle last epoch case if too short
            if epoch_end_sample > len(audio):
                break
            
            epoch_audio = audio[epoch_start_sample:epoch_end_sample]
            
            # Quality Check
            quality_metrics = self.check_audio_quality(epoch_audio)
            quality_score = quality_metrics['quality_score']
            
            if quality_score < self.min_audio_quality:
                label_counts[-2] += 1
                continue
            
            # Actual start and end time per recording start
            epoch_start_time = record_start + (i * self.epoch_duration)
            epoch_end_time = epoch_start_time + self.epoch_duration
            
            label = self.extract_epoch_labels(epoch_start_time, epoch_end_time, events, awake_intervals)
            
            # Don't care if awake
            if label == -1:
                label_counts[-1] += 1
                continue
            
            epoch_array.append(epoch_audio)
            label_array.append(label)
            quality_array.append(quality_score)
            label_counts[label] += 1
        
        epoch_array = np.array(epoch_array, dtype=np.float32)
        label_array = np.array(label_array, dtype=np.int32)
        quality_array = np.array(quality_array, dtype=np.float32)
        
        # Extract features if enabled
        if self.extract_features:
            print(f"    Extracting AST features...")
            epoch_array = self.extract_features_batch(epoch_array, apply_normalization=True)
            print(f"    Feature shape: {epoch_array.shape}")
        
        print(f"\nEpoch Statistics for folder {folder_id}:")
        print(f"    Total Epochs: {num_epochs}")
        print(f"    Processed Epochs Saved: {len(label_array)}")
        print(f"    Low Quality Skipped: {label_counts[-2]}")
        print(f"    Awake Epochs Skipped: {label_counts[-1]}")
        print(f"    No Event Epochs: {label_counts[0]}")
        print(f"    OSA Event Epochs: {label_counts[1]}")
        print(f"    Hypopnea Event Epochs: {label_counts[2]}")
        print(f"    Mean Quality Score: {np.mean(quality_array):.3f}")
        
        # Clear audio from memory
        del audio
        gc.collect()
        
        return epoch_array, label_array, quality_array
    
    def _save_index(self):
        folder_metadata = {}
        total_epochs = 0
        class_distribution = {0: 0, 1: 0, 2: 0}
    
        files = sorted(self.output_dir.glob("folder_*.pkl*"))
    
        # Create index structure
        index_data = {
            'folder_metadata': self.folder_metadata,
            'preprocessing_stats': self.preprocessing_stats,
        }
        
        for file_path in files:
            folder_id = file_path.stem.replace("folder_", "").replace(".pkl", "")

            # Load lightweight metadata
            with (gzip.open(file_path, "rb") if file_path.suffix == ".gz" else open(file_path, "rb")) as f:
                data = pickle.load(f)

            labels = np.array(data["labels"])
            num_epochs = len(labels)

            # Determine global start/end indices
            start_idx = total_epochs
            end_idx = start_idx + num_epochs

            # Count per-class
            label_counts = {
                0: int(np.sum(labels == 0)),
                1: int(np.sum(labels == 1)),
                2: int(np.sum(labels == 2)),
            }

            # Add to totals
            total_epochs = end_idx
            for lbl, cnt in label_counts.items():
                class_distribution[lbl] += cnt

            # Store folder metadata
            folder_metadata[folder_id] = {
                "file_path": file_path.name,
                "num_epochs": num_epochs,
                "start_idx": start_idx,
                "end_idx": end_idx,
                "label_distribution": label_counts,
                "mean_quality": float(np.mean(data["quality_scores"])),
                "processed_timestamp": data.get("timestamp", None),
            }

        # Save global stats
        preprocessing_stats = {
            "total_folders": len(folder_metadata),
            "total_epochs": total_epochs,
            "class_distribution": class_distribution,
            "config": self.preprocessing_stats["config"],
        }

        # Construct final index structure
        index_data = {
            "folder_metadata": folder_metadata,
            "preprocessing_stats": preprocessing_stats,
        }
        
        self.folder_metadata = folder_metadata
        self.preprocessing_stats = preprocessing_stats

        # Write to disk
        with open(self.index_path, "w") as f:
            json.dump(index_data, f, indent=2)

        print("\nIndex rebuilt and saved!")
        print(f"    Total folders: {preprocessing_stats['total_folders']}")
        print(f"    Total epochs:  {preprocessing_stats['total_epochs']:,}")
        print("    Class distribution:")
        for label, count in class_distribution.items():
            name = {0: "No Event", 1: "OSA", 2: "Hypopnea"}[label]
            print(f"       {name}: {count}")
    
    def save_folder_individually(self, folder_id, epochs, labels, quality_scores=None):
        file_ext = ".pkl.gz" if self.use_compression else ".pkl"
        folder_output_path = self.output_dir / f"folder_{folder_id}{file_ext}"
        
        # Count labels
        label_counts = {}
        for label in [0, 1, 2]:
            label_counts[label] = int(np.sum(labels == label))
        
        data = {
            'epochs': epochs,
            'labels': labels,
            'quality_scores': quality_scores if quality_scores is not None else np.ones(len(labels)),
            'folder_id': folder_id,
            'sample_rate': self.sample_rate,
            'epoch_duration': self.epoch_duration,
            'overlap_threshold': self.overlap_threshold,
        }
        
        # Use highest protocol for better compression and speed
        if self.use_compression:
            with gzip.open(folder_output_path, 'wb', compresslevel=6) as f:
                pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
        else:
            with open(folder_output_path, 'wb') as f:
                pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
        
        file_size_mb = folder_output_path.stat().st_size / (1024 * 1024)
        print(f"    Saved to {folder_output_path} ({file_size_mb:.2f} MB)")
        
        return folder_output_path
    
    def process_folders(self, folder_ids: List[str] = None):
        # If no folder IDs provided, process all folders in data_dir
        if (not folder_ids):
            print ("\nNo folder IDs provided. Defaulting to folders 01-50.")
            folder_ids = [f"{i:02d}" for i in range(1, 51)] # Folders named 01 to 50
        
        print(f"\n{'='*40}")
        print(f"Processing {len(folder_ids)} folders")
        print(f"{'='*40}")
        
        processed_count = 0
        
        for folder_id in folder_ids:
            output_pkl = self.output_dir / f"folder_{folder_id}.pkl"
            output_gz = self.output_dir / f"folder_{folder_id}.pkl.gz"

            if output_pkl.exists() or output_gz.exists():
                print(f"\nSkipping folder {folder_id} (output file already exists)")
                continue
                
            folder_path = self.data_dir / folder_id
            audio_path = folder_path / f"{folder_id}_phone.wav"
            annotation_path = folder_path / f"{folder_id}_annotation.json"
            
            if not (folder_path.exists() and audio_path.exists() and annotation_path.exists()):
                print(f"\nSkipping folder {folder_id}: Missing files")
                continue
            
            print(f"\nProcessing folder {folder_id}...")
            
            try:
                epochs, labels, quality_scores = self.create_epochs(folder_id)
                
                # Save this folder's data immediately
                print(f"\nSaving folder {folder_id} data...")
                self.save_folder_individually(folder_id, epochs, labels, quality_scores)
                
                processed_count += 1
                print(f"Completed folder {folder_id} ({processed_count} folders processed)")
                
            except Exception as e:
                print(f"    Error processing folder {folder_id}: {e}")
                import traceback
                traceback.print_exc()
                continue
            finally:
                # Always clear memory after each folder
                del epochs, labels
                gc.collect()
        
        # After all folders are processed
        if processed_count > 0:
            print(f"\n{processed_count} new folders processed.")
            self._save_index()
        else:
            print("\nNo new folders were processed!")

if __name__ == "__main__":
    preprocessor = AudioPreprocessor(
        data_dir=config.DATA_DIR,
        output_dir=config.PREPROCESSED_DIR,
        epoch_duration=config.AUDIO_EPOCH_DURATION,
        sample_rate=config.AUDIO_SAMPLE_RATE,
        use_compression=config.USE_COMPRESSION,
        extract_features=config.USE_PREEXTRACTED_FEATURES,
        overlap_threshold=config.OVERLAP_THRESHOLD,
        min_audio_quality=config.MIN_AUDIO_QUALITY
    )
    
    folder_ids = [f"{i:02d}" for i in range(1, config.NUM_AUDIO_FILES+1)]
    
    data = preprocessor.process_folders(folder_ids=folder_ids)

c:\Users\jacst\anaconda3\envs\AST\Lib\site-packages\torch\nn\modules\module.py:1093: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  return self._apply(lambda t: t.cuda(device))


    Feature extraction will use GPU

Initialized AudoPreprocessor
    Data_dir: c:\Users\jacst\Downloads\School\Fall25\CSE575/Data
    Output_dir: c:\Users\jacst\Downloads\School\Fall25\CSE575/Preprocessed
    Epoch Duration: 10s
    Sample Rate: 16000Hz
    Samples per epoch: 160000
    Compression: Enabled (gzip)
    Overlap Threshold: 50%
    Min Audio Quality: 0.3

Processing 50 folders

Skipping folder 01 (output file already exists)

Skipping folder 02 (output file already exists)

Skipping folder 03 (output file already exists)

Skipping folder 04 (output file already exists)

Skipping folder 05 (output file already exists)

Skipping folder 06 (output file already exists)

Skipping folder 07 (output file already exists)

Skipping folder 08 (output file already exists)

Skipping folder 09 (output file already exists)

Skipping folder 10 (output file already exists)

Skipping folder 11: Missing files

Skipping folder 12 (output file already exists)

Skipping folder 13 (output file

In [2]:
class MultiEpochSleepApneaDetector(nn.Module):
    # Apnea Detector using Audio Spectrogram Transformer (AST)
    
    # Studies 14->10 Architecture
    # 14 contextual epochs to predict 10 output epochs
    def __init__(self, context_epochs=14, output_epochs=10, num_classes=3, dropout=0.3, lstm_enabled=False, 
                 lstm_layers=2, transformer_encoder_enabled=False, transformer_layers=6,
                use_preextracted_features=True):

        super().__init__()
        
        self.context_epochs = context_epochs
        self.output_epochs = output_epochs
        self.num_classes = num_classes
        self.lstm_enabled = lstm_enabled
        self.lstm_layers = lstm_layers
        self.transformer_encoder_enabled = transformer_encoder_enabled
        self.transformer_layers = transformer_layers
        self.use_preextracted_features = use_preextracted_features
        
        print(f"\nInitializing MultiEpochSleepApneaDetector: ")
        print(f"    Context Epochs: {context_epochs}")
        print(f"    Output Epochs: {output_epochs}")
        print(f"    Number of Classes: {num_classes}")
        print(f"    Dropout: {dropout}")
        print(f"    Pre-extracted Features: {use_preextracted_features}")

        if not use_preextracted_features:
            # Load pre-trained AST model for on-the-fly extraction
            self.ast_config = ASTConfig.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
            self.ast_model = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
            self.ast_feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
            
            self.ast_model.gradient_checkpointing_enable()
            ast_feature_dim = self.ast_config.hidden_size #768
        else:
            # Features already extracted, just use them
            ast_feature_dim = 768  # AST hidden size

        self.hybrid_mode = lstm_enabled and transformer_encoder_enabled
        
        if self.hybrid_mode:
            print(f"    Using LSTM -> Positional Transformer Architecture")
            print(f"    LSTM Layers: {self.lstm_layers}")
            print(f"    Transformer Layers: {self.transformer_layers}")
            self.temporal_lstm = nn.LSTM(
                input_size=ast_feature_dim,
                hidden_size=512,
                num_layers=self.lstm_layers,
                batch_first=True,
                dropout=dropout if context_epochs > 1 else 0,
                bidirectional=True
            )

            lstm_output_dim = 1024   # bidirectional 512×2

            self.lstm_layernorm = nn.LayerNorm(lstm_output_dim)
            self.proj = nn.Linear(lstm_output_dim, ast_feature_dim)

            encoder_layer = nn.TransformerEncoderLayer(
                d_model=ast_feature_dim,
                nhead=8,
                dim_feedforward=2048,
                dropout=dropout,
                batch_first=True
            )
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=self.transformer_layers)

            self.positional_encoding = nn.Parameter(
                torch.randn(1, context_epochs, ast_feature_dim)
            )
            
            classifier_input_dim = ast_feature_dim
        elif transformer_encoder_enabled:
            print(f"    Using Positional Transformer Encoder Architecture")
            print(f"    Transformer Layers: {self.transformer_layers}")
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=ast_feature_dim,
                nhead=8,
                dim_feedforward=2048,
                dropout=dropout,
                batch_first=True
            )
            self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=self.transformer_layers)

            self.positional_encoding = nn.Parameter(
                torch.randn(1, context_epochs, ast_feature_dim)
            )

            classifier_input_dim = ast_feature_dim
        elif lstm_enabled:
            print(f"    Using LSTM Architecture")
            print(f"    LSTM Layers: {self.lstm_layers}")
            self.temporal_lstm = nn.LSTM(
                input_size=ast_feature_dim,
                hidden_size=512,
                num_layers=self.lstm_layers,
                batch_first=True,
                dropout=dropout if context_epochs > 1 else 0,
                bidirectional=True
            )

            lstm_output_dim = 1024

            self.lstm_layernorm = nn.LayerNorm(lstm_output_dim)

            # Attention layer
            self.attention = nn.MultiheadAttention(
                embed_dim=lstm_output_dim,
                num_heads=8,
                dropout=dropout,
                batch_first=True
            )
            
            classifier_input_dim = lstm_output_dim
        else:
            print(f"    Using Non-Temporal Architecture")
            classifier_input_dim = ast_feature_dim

        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_dim, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(dropout),

            nn.Linear(512, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.GELU(),
            nn.LayerNorm(256),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.GELU(),
            nn.LayerNorm(128),
            nn.Dropout(dropout),

            nn.Linear(128, num_classes),
        )
        
        print(f"\nModel initialized successfully.")
        print(f"AST Feature Dimension: {ast_feature_dim}")
        
    def extract_ast_features(self, waveforms):
        # Only used if features are not pre-extracted
        batch_size, n_epochs, n_samples = waveforms.shape
        waveforms_flat = waveforms.reshape(batch_size * n_epochs, n_samples)
        
        inputs = self.ast_feature_extractor(
            [w.cpu().numpy() for w in waveforms_flat], 
            sampling_rate=16000, 
            return_tensors="pt"
        )
        
        inputs = {k: v.to(next(self.parameters()).device) for k, v in inputs.items()}
        
        outputs = self.ast_model(**inputs)
        features = outputs.last_hidden_state[:, 0, :]
            
        features = features.reshape(batch_size, n_epochs, -1)
        return features
    
    def forward(self, input_data):
        # input_data is either raw waveforms or pre-extracted features
        if self.use_preextracted_features:
            # Input is already features: (batch, epochs, 768)
            features = input_data
        else:
            # Extract features from raw audio
            features = self.extract_ast_features(input_data)
            
        if self.hybrid_mode:
            # LSTM
            lstm_out, _ = self.temporal_lstm(features)
            lstm_out = self.lstm_layernorm(lstm_out) # (B, 14, 1024)

            proj_out = self.proj(lstm_out) # (B, 14, 768)

            # Add positional encoding
            positional_out = proj_out + self.positional_encoding[:, :proj_out.size(1), :]

            # Transformer
            trans_out = self.transformer(positional_out)

            start = (self.context_epochs - self.output_epochs) // 2
            end = start + self.output_epochs
            output_features = trans_out[:, start:end, :]
        elif self.transformer_encoder_enabled:
            # Input shape: (batch, epochs, 768)
            features_positional = features + self.positional_encoding[:, :features.size(1), :]
            transformer_out = self.transformer(features_positional)
        
            start_idx = (self.context_epochs - self.output_epochs) // 2
            end_idx = start_idx + self.output_epochs
            output_features = transformer_out[:, start_idx:end_idx, :]
        elif self.lstm_enabled:
            lstm_out, _ = self.temporal_lstm(features)
            lstm_out = self.lstm_layernorm(lstm_out)
            
            attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
            
            start_idx = (self.context_epochs - self.output_epochs) // 2
            end_idx = start_idx + self.output_epochs
            output_features = attn_out[:, start_idx:end_idx, :]
        else:
            start_idx = (self.context_epochs - self.output_epochs) // 2
            end_idx = start_idx + self.output_epochs
            output_features = features[:, start_idx:end_idx, :]
        
        logits = self.classifier(output_features)
        return logits
    
class FocalLoss(nn.Module):
    def __init__ (self, alpha=None, gamma=2.0, label_smoothing=0.0, minority_gradient_scale=2.0):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.minority_gradient_scale = minority_gradient_scale

        if alpha is not None:
            if isinstance(alpha, (list, np.ndarray)):
                alpha = torch.tensor(alpha, dtype=torch.float32)
            self.alpha = alpha
        else:
            self.alpha = None
        
        print(f"\nInitialized FocalLoss:")
        print(f"    Gamma (focusing): {gamma}")
        print(f"    Alpha (class weights): {alpha}")
        print(f"    Label Smoothing: {label_smoothing}")
        print(f"    Minority Gradient Scale: {minority_gradient_scale}")
        
    def forward(self, logits, targets):
        # logits: (batch, n_epochs, n_classes)
        # targets: (batch, n_epochs)

        # Reshape to (batch*n_epochs, n_classes) and (batch*n_epochs)
        batch_size, n_epochs, n_classes = logits.shape
        logits_flat = logits.reshape(-1, n_classes)
        targets_flat = targets.reshape(-1)
        
        if self.alpha is not None:
            alpha = self.alpha.to(logits.device)
        else:
            alpha = None
        
        # Calculate softmax probabilities
        probs = F.softmax(logits_flat, dim=1)
        
        # Get probability of correct class
        targets_one_hot = F.one_hot(targets_flat, num_classes=n_classes).float()

        # Label Smoothing
        if self.label_smoothing > 0:
            targets_one_hot = targets_one_hot * (1 - self.label_smoothing) + self.label_smoothing / n_classes
        
        pt = (probs * targets_one_hot).sum(dim=1) 
        
        # Calculate focal term: (1 - pt)^gamma
        focal_weight = (1 - pt) ** self.gamma
        
        # Calculate cross entropy
        ce_loss = F.cross_entropy(
            logits_flat, 
            targets_flat, 
            weight=alpha,
            reduction='none'
        )
        
        # Combine: focal_weight * ce_loss
        focal_loss = focal_weight * ce_loss
        
        minority_mask = (targets_flat == 1) | (targets_flat == 2)
        if minority_mask.any() and self.training:
            # Create gradient scaling
            scale = torch.ones_like(focal_loss)
            scale[minority_mask] = self.minority_gradient_scale
            focal_loss = focal_loss * scale
        
        return focal_loss.mean()
    
class SleepApneaDataset(Dataset):
    def __init__(self, preprocessed_dir, context_epochs=14, output_epochs=10, 
                 use_compression=True, cache_size=3):
        
        self.preprocessed_dir = Path(preprocessed_dir)
        self.context_epochs = context_epochs
        self.output_epochs = output_epochs
        self.use_compression = use_compression
        self.cache_size = cache_size
        
        self.folder_cache = {}
        self.cache_order = []
        
        index_path = self.preprocessed_dir / "index.json"
        
        if not index_path.exists():
            raise FileNotFoundError(
                f"Index not found at {index_path}. "
                f"Please run preprocessing first!"
            )
        
        print(f"\nLoading dataset index from {index_path}...")
        with open(index_path, 'r') as f:
            index_data = json.load(f)
        
        # Load folder metadata
        self.folder_metadata = {
            k: {
                **v,
                'file_path': self.preprocessed_dir / v['file_path']
            }
            for k, v in index_data['folder_metadata'].items()
        }
        
        self.preprocessing_stats = index_data['preprocessing_stats']
        
        # Build valid sequence indices
        self.valid_indices = self._build_sequence_indices()
        
        print(f"\nInitialized SleepApneaDataset:")
        print(f"    Total folders: {len(self.folder_metadata)}")
        print(f"    Total epochs: {self.preprocessing_stats['total_epochs']:,}")
        print(f"    Valid sequences: {len(self.valid_indices):,}")
        print(f"    Context epochs: {self.context_epochs}")
        print(f"    Output epochs: {self.output_epochs}")
        print(f"    Cache size: {self.cache_size} folders")
    
    def _build_sequence_indices(self):
        # Build valid sequence indices based on context_epochs
        valid_indices = []
        
        for folder_id, metadata in self.folder_metadata.items():
            folder_num_epochs = metadata['num_epochs']
            folder_start = metadata['start_idx']
            
            # Calculate how many valid sequences in this folder
            num_sequences = folder_num_epochs - self.context_epochs + 1
            
            if num_sequences <= 0:
                print(f"Warning: Folder {folder_id} has only {folder_num_epochs} epochs, "
                      f"need {self.context_epochs} for context. Skipping.")
                continue
            
            for i in range(num_sequences):
                valid_indices.append({
                    'global_idx': folder_start + i,
                    'folder_id': folder_id,
                    'local_idx': i
                })
        
        return valid_indices
    
    def _load_folder(self, folder_id):
        if folder_id in self.folder_cache:
            return self.folder_cache[folder_id]
        
        file_path = self.folder_metadata[folder_id]['file_path']
        
        if self.use_compression:
            import gzip
            import pickle
            with gzip.open(file_path, 'rb') as f:
                data = pickle.load(f)
        else:
            import pickle
            with open(file_path, 'rb') as f:
                data = pickle.load(f)
        
        # Cache management
        if len(self.folder_cache) >= self.cache_size:
            oldest_folder = self.cache_order.pop(0)
            del self.folder_cache[oldest_folder]
        
        self.folder_cache[folder_id] = {
            'epochs': data['epochs'],
            'labels': data['labels']
        }
        self.cache_order.append(folder_id)
        
        return self.folder_cache[folder_id]
    
    def _get_folder_split_indices(self, train_folders, val_folders):
        train_indices = []
        val_indices = []
        
        for idx, seq_info in enumerate(self.valid_indices):
            folder_id = seq_info['folder_id']
            if folder_id in train_folders:
                train_indices.append(idx)
            elif folder_id in val_folders:
                val_indices.append(idx)
        
        return train_indices, val_indices
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, idx):
        import torch
        
        sequence_info = self.valid_indices[idx]
        folder_id = sequence_info['folder_id']
        local_idx = sequence_info['local_idx']
        
        folder_data = self._load_folder(folder_id)
        
        # Extract sequence
        start_idx = local_idx
        context = folder_data['epochs'][start_idx:start_idx + self.context_epochs]
        
        label_start = start_idx + (self.context_epochs - self.output_epochs) // 2
        label_end = label_start + self.output_epochs
        labels = folder_data['labels'][label_start:label_end]
        
        context_tensor = torch.from_numpy(context).float()
        labels_tensor = torch.from_numpy(labels).long()
        
        return context_tensor, labels_tensor
    
class Trainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, device, output_dir, scheduler = None, 
                 patience=10, gradient_accumulation_steps=1, use_mixed_precision=True):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.output_dir = Path(output_dir)
        self.patience = patience
        self.gradient_accumulation_steps = gradient_accumulation_steps
        self.use_mixed_precision = use_mixed_precision
        
        # Initialize gradient scaler for mixed precision
        self.scaler = GradScaler(device=self.device.type) if use_mixed_precision else None
        
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        # History
        self.history = {
            'learning_rate': [],
            'train_loss': [],
            'train_accuracy': [],
            'val_loss': [],
            'val_accuracy': [],
            'val_f1': [],
            'minority_f1': [], # Avg of Hypo and OSA F1
            'weighted_f1': [] # Weighted F1 of all classes per config
        }
        
        self.best_val_f1 = 0.0
        self.best_minority_f1 = 0.0
        self.best_weighted_f1 = 0.0
        self.best_epoch = 0
        self.epochs_no_improve = 0
        
        # Stats Monitoring
        if torch.cuda.is_available():
            try:
                pynvml.nvmlInit()
                self.nvml_handle = pynvml.nvmlDeviceGetHandleByIndex(0)
                self.use_nvml = True
            except:
                self.use_nvml = False
                print("Warning: Could not initialize NVML for detailed GPU stats")
        else:
            self.use_nvml = False
        
        print(f"\nInitialized Trainer:")
        print(f"    Device: {self.device}")
        print(f"    Output Directory: {self.output_dir}")
        print(f"    Training Batches: {len(self.train_loader)}")
        print(f"    Val Batches: {len(self.val_loader)}")
        print(f"    Patience: {self.patience} epochs")
        print(f"    Mixed Precision: {self.use_mixed_precision}")
        print(f"    Gradient Accumulation: {self.gradient_accumulation_steps}")
    
    def get_gpu_stats(self):
        if not torch.cuda.is_available():
            return {}
        
        stats = {}
        
        # Memory stats
        stats['mem_alloc'] = torch.cuda.memory_allocated() / 1024**3
        stats['mem_reserved'] = torch.cuda.memory_reserved() / 1024**3
        
        # Detailed stats with NVML
        if self.use_nvml:
            try:
                mem_info = pynvml.nvmlDeviceGetMemoryInfo(self.nvml_handle)
                stats['mem_used'] = mem_info.used / 1024**3
                stats['mem_total'] = mem_info.total / 1024**3
                
                util = pynvml.nvmlDeviceGetUtilizationRates(self.nvml_handle)
                stats['gpu_util'] = util.gpu
                
                temp = pynvml.nvmlDeviceGetTemperature(self.nvml_handle, pynvml.NVML_TEMPERATURE_GPU)
                stats['temp'] = temp
            except:
                pass
        
        return stats
    
    def train_epoch(self, epoch):
        self.model.train()
        
        total_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch} [Training]", leave=False)
        self.optimizer.zero_grad()
        
        for batch_idx, (data, targets) in enumerate(pbar):
            data, targets = data.to(self.device), targets.to(self.device)

            # Augment Minority Classes With Mask
            if epoch > 3:
                minority_mask = (targets[:, 0] == 1) | (targets[:,0] == 2)
                if minority_mask.any():
                    if np.random.rand() < 0.4:
                        noise = torch.randn_like(data[minority_mask]) * 0.01
                        data[minority_mask] = data[minority_mask] + noise
                    if np.random.rand() < 0.3:
                        dropout_mask = torch.rand_like(data[minority_mask]) > 0.1
                        data[minority_mask] = data[minority_mask] * dropout_mask
                    if np.random.rand() < 0.2 and data[minority_mask].size(1) > 2:
                        shift_amount = np.random.randint(-2, 3)
                        data[minority_mask] = torch.roll(data[minority_mask], shifts=shift_amount, dims=1)
                    if np.random.rand() < 0.3:
                        scale_factor = np.random.uniform(0.9, 1.1)
                        data[minority_mask] = data[minority_mask] * scale_factor
            
            # Mixed precision training
            if self.use_mixed_precision:
                with autocast(device_type=self.device.type):
                    logits = self.model(data)
                    loss = self.criterion(logits, targets) / self.gradient_accumulation_steps
                
                self.scaler.scale(loss).backward()
            else:
                logits = self.model(data)
                loss = self.criterion(logits, targets) / self.gradient_accumulation_steps
                loss.backward()
            
            # Update weights every N steps
            if (batch_idx + 1) % self.gradient_accumulation_steps == 0:
                if self.use_mixed_precision:
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=config.MAX_GRAD_NORM)
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=config.MAX_GRAD_NORM)
                    self.optimizer.step()
                
                self.optimizer.zero_grad()
            
            # Statistics
            total_loss += loss.item() * self.gradient_accumulation_steps
            predictions = logits.argmax(dim=-1)
            correct += (predictions == targets).sum().item()
            total += targets.numel()
            
            # Update Dict
            postfix_dict = {
                'Loss': f"{total_loss/(batch_idx+1):.4f}",
                'Acc': f"{100.0 * correct / total:.2f}%"
            }
            
            # Add GPU/Memory stats
            gpu_stats = self.get_gpu_stats()
            if gpu_stats:
                postfix_dict['GPU_Mem'] = f"{gpu_stats['mem_alloc']:.1f}GB"
                if 'gpu_util' in gpu_stats:
                    postfix_dict['GPU%'] = f"{gpu_stats['gpu_util']}%"
                if 'temp' in gpu_stats:
                    postfix_dict['Temp'] = f"{gpu_stats['temp']}°C"
            
            # System RAM
            postfix_dict['RAM'] = f"{psutil.virtual_memory().percent:.1f}%"
            
            pbar.set_postfix(postfix_dict)
            
            # Clear cache periodically
            if batch_idx % 100 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        # Handle remaining gradients
        if (batch_idx + 1) % self.gradient_accumulation_steps != 0:
            if self.use_mixed_precision:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=config.MAX_GRAD_NORM)
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=config.MAX_GRAD_NORM)
                self.optimizer.step()
            
            self.optimizer.zero_grad()
        
        avg_loss = total_loss / len(self.train_loader)
        accuracy = 100.0 * correct / total
        
        return avg_loss, accuracy
    
    def validate_epoch(self, epoch):
        self.model.eval()
        
        total_loss = 0.0
        correct = 0
        total = 0

        # Per-class tracking
        class_correct = {0: 0, 1: 0, 2: 0}  # no_event, osa, hypo
        class_total = {0: 0, 1: 0, 2: 0}
        
        all_predictions = []
        all_targets = []
        
        pbar = tqdm(self.val_loader, desc=f"Epoch {epoch} [Validation]", leave=False)
        
        with torch.no_grad():
            for data, targets in pbar:
                data, targets = data.to(self.device), targets.to(self.device)
                
                # Mixed precision for validation too
                if self.use_mixed_precision:
                    with autocast(device_type=self.device.type):
                        logits = self.model(data)
                        loss = self.criterion(logits, targets)
                else:
                    logits = self.model(data)
                    loss = self.criterion(logits, targets)
                
                # Statistics
                total_loss += loss.item()
                predictions = logits.argmax(dim=-1)
                correct += (predictions == targets).sum().item()
                total += targets.numel()

                # Per-class statistics
                for class_idx in [0, 1, 2]:
                    class_mask = (targets == class_idx)
                    class_total[class_idx] += class_mask.sum().item()
                    class_correct[class_idx] += ((predictions == targets) & class_mask).sum().item()
                
                # Update Dict
                postfix_dict = {
                    'Loss': f"{total_loss/(pbar.n+1):.4f}",
                    'Acc': f"{100.0 * correct / total:.2f}%"
                }
                
                # Add GPU/Memory stats
                gpu_stats = self.get_gpu_stats()
                if gpu_stats:
                    postfix_dict['GPU_Mem'] = f"{gpu_stats['mem_alloc']:.1f}GB"
                    if 'gpu_util' in gpu_stats:
                        postfix_dict['GPU%'] = f"{gpu_stats['gpu_util']}%"
                    if 'temp' in gpu_stats:
                        postfix_dict['Temp'] = f"{gpu_stats['temp']}°C"
                
                # System RAM
                postfix_dict['RAM'] = f"{psutil.virtual_memory().percent:.1f}%"

                # Progres Update
                pbar.set_postfix(postfix_dict)
                
                all_predictions.extend(predictions.cpu().numpy().flatten())
                all_targets.extend(targets.cpu().numpy().flatten())
        
        avg_loss = total_loss / len(self.val_loader)
        accuracy = 100.0 * correct / total
        
        # Calculate F1 Scores
        f1_macro = f1_score(all_targets, all_predictions, average='macro', zero_division=0)
        f1_per_class = f1_score(all_targets, all_predictions, average=None, zero_division=0)
        minority_f1 = (f1_per_class[1] + f1_per_class[2]) / 2
        weighted_f1 = (config.F1_WEIGHTS[0] * f1_per_class[0] + 
                       config.F1_WEIGHTS[1] * f1_per_class[1] +
                       config.F1_WEIGHTS[2] * f1_per_class[2])

        # Calculate per-class accuracy
        class_names = {0: 'No Event', 1: 'OSA', 2: 'Hypopnea'}
        per_class_acc = {}
        for class_idx in [0, 1, 2]:
            if class_total[class_idx] > 0:
                per_class_acc[class_idx] = 100.0 * class_correct[class_idx] / class_total[class_idx]
            else:
                per_class_acc[class_idx] = 0.0

        # Metrics
        metrics = {
            'loss': avg_loss,
            'accuracy': accuracy,
            'f1_macro': f1_macro,
            'weighted_f1': weighted_f1,
            'minority_f1': minority_f1,
            'class_metrics': {
                class_names[i]: {
                    'accuracy': per_class_acc[i],
                    'f1': f1_per_class[i],
                    'count': class_total[i]
                }
                for i in [0, 1, 2]
            }
        }
        
        return metrics, all_predictions, all_targets
    
    def train(self, num_epochs):
        print("\n" + "="*40)
        print(f"Starting training for {num_epochs} epochs...")
        print("="*40)
        
        for epoch in range(1, num_epochs + 1):
            print(f"\nEpoch {epoch}/{num_epochs}")
            print("-"*40)

            current_lr = self.optimizer.param_groups[0]['lr']
            print(f"Learning Rate: {current_lr:.4e}")
            
            train_loss, train_acc = self.train_epoch(epoch)
            val_metrics, _, _ = self.validate_epoch(epoch)

            if self.scheduler is not None:
                #self.scheduler.step(val_metrics['loss'])
                self.scheduler.step() # Per Epoch
            
            # Log history
            self.history['learning_rate'].append(current_lr)
            self.history['train_loss'].append(train_loss)
            self.history['train_accuracy'].append(train_acc)
            self.history['val_loss'].append(val_metrics['loss'])
            self.history['val_accuracy'].append(val_metrics['accuracy'])
            self.history['val_f1'].append(val_metrics['f1_macro'])
            self.history['weighted_f1'].append(val_metrics['weighted_f1'])
            self.history['minority_f1'].append(val_metrics['minority_f1'])

            # Class history
            if 'per_class_accuracy' not in self.history:
                self.history['per_class_accuracy'] = {
                    'No Event': [],
                    'OSA': [],
                    'Hypopnea': []
                }
                self.history['per_class_f1'] = {
                    'No Event': [],
                    'OSA': [],
                    'Hypopnea': []
                }

            for class_name, metrics in val_metrics['class_metrics'].items():
                self.history['per_class_accuracy'][class_name].append(metrics['accuracy'])
                self.history['per_class_f1'][class_name].append(metrics['f1'])

            print(f"\nEpoch {epoch} Summary:")
            print(f"    Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
            print(f"    Val Loss: {val_metrics['loss']:.4f}, Val Acc: {val_metrics['accuracy']:.2f}%")
            print(f"    Val F1 (Macro): {val_metrics['f1_macro']:.4f}")
            print(f"    Val F1 (Weighed): {val_metrics['weighted_f1']:.4f}")
            print(f"    Val F1 (Minority): {val_metrics['minority_f1']:.4f}")
            
            print(f"\n    Per-Class Metrics:")
            print(f"    {'Class':<15} {'Count':<8} {'Accuracy':<12} {'F1 Score':<10}")
            print(f"    {'-'*50}")
            for class_name, metrics in val_metrics['class_metrics'].items():
                print(f"    {class_name:<15} {metrics['count']:<8} {metrics['accuracy']:>6.2f}%      {metrics['f1']:>6.4f}")
        
            
            # Check for improvement
            if val_metrics['weighted_f1'] > self.best_weighted_f1:
                self.best_weighted_f1 = val_metrics['weighted_f1']
                self.best_minority_f1 = val_metrics['minority_f1']
                self.best_val_f1 = val_metrics['f1_macro']
                self.best_epoch = epoch
                self.best_metrics = val_metrics.copy()
                self.epochs_no_improve = 0
                
                checkpoint_path = self.output_dir / "best_model.pth"
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_metrics': val_metrics,
                    'best_metrics': self.best_metrics  # NEW: Save in checkpoint
                }, checkpoint_path)
                
                print(f"    New best model saved (Weighted F1: {val_metrics['weighted_f1']:.4f})")
            else:
                self.epochs_no_improve += 1
                print(f"    No improvement for {self.epochs_no_improve} epochs.")
            
            
            # Early stopping
            if self.epochs_no_improve >= self.patience:
                print(f"\nEarly stopping triggered after {self.patience} epochs with no improvement.")
                print(f"Best model from epoch {self.best_epoch} with Weighted Val F1: {self.best_weighted_f1:.4f}")
                break
            
            # Periodic checkpoints
            if epoch % 5 == 0:
                checkpoint_path = self.output_dir / f'checkpoint_epoch_{epoch}.pth'
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'history': self.history
                }, checkpoint_path)
                print(f"    Checkpoint saved to checkpoint_epoch_{epoch}.pth")
        
        # Save training history
        history_path = self.output_dir / "training_history.json"
        with open(history_path, 'w') as f:
            json.dump(self.history, f, indent=2)
        print(f"\nTraining history saved to {history_path}")
        
        print(f"\n{'='*40}")
        print(f"Training Complete!")
        print(f"   Total epochs: {epoch}")
        print(f"Best Epoch: {self.best_epoch}")
        print(f"Best Macro F1: {self.best_val_f1:.4f}")
        print(f"Best Weighted F1: {self.best_weighted_f1:.4f}")
        print(f"Best Minority F1: {self.best_minority_f1:.4f}")
        print(f"\nBest Epoch Per-Class F1 Scores:")
        if self.best_metrics:
            for class_name, metrics in self.best_metrics['class_metrics'].items():
                print(f"    {class_name}: {metrics['f1']:.4f}")
        print(f"{'='*40}")
        
if __name__ == "__main__":
    print("\n" + "="*60)
    print("Sleep Apnea Detection Metadata Prep")
    print("="*60)
    
    # Create Dataset
    print(f"\nLoading preprocessed data from {config.PREPROCESSED_DIR}...")
    dataset = SleepApneaDataset(
        preprocessed_dir=config.PREPROCESSED_DIR,
        context_epochs=config.CONTEXT_EPOCHS,
        output_epochs=config.OUTPUT_EPOCHS,
        use_compression=config.USE_COMPRESSION,
        cache_size=config.CACHE_SIZE
    )


Sleep Apnea Detection Metadata Prep

Loading preprocessed data from /home/jwethere/Preprocessed...

Loading dataset index from /home/jwethere/Preprocessed/index.json...

Initialized SleepApneaDataset:
    Total folders: 48
    Total epochs: 138,943
    Valid sequences: 138,031
    Context epochs: 20
    Output epochs: 10
    Cache size: 48 folders


In [ ]:
# Primary Code
def create_balanced_sampler(dataset, indices):
    class_counts = {0: 0, 1: 0, 2: 0}
    
    print("\nCalculating class distribution for balanced sampling...")
    for idx in tqdm(indices, desc="Counting classes"):
        # Handle both regular Dataset and Subset
        if isinstance(dataset, Subset):
            _, labels = dataset.dataset[dataset.indices[idx]]
        else:
            _, labels = dataset[idx]
        
        for label in labels.numpy():
            class_counts[int(label)] += 1
            
    total = sum(class_counts.values())
    
    # Calculate weights (inverse frequency) ^ 0.70
    class_weights = {
        cls: (total / count) ** 0.70
        for cls, count in class_counts.items()
    }
    
    print(f"\nClass distribution:")
    print(f"    No Event: {class_counts[0]:,} ({class_counts[0]/total*100:.1f}%)")
    print(f"    OSA:      {class_counts[1]:,} ({class_counts[1]/total*100:.1f}%)")
    print(f"    Hypopnea: {class_counts[2]:,} ({class_counts[2]/total*100:.1f}%)")
    print(f"\nSampling weights: {[round(class_weights[i], 2) for i in [0, 1, 2]]}")
    
    # Assign weight to each sample
    sample_weights = []
    for idx in indices:
        if isinstance(dataset, Subset):
            _, labels = dataset.dataset[dataset.indices[idx]]
        else:
            _, labels = dataset[idx]
        
        weights = [class_weights[int(l)] for l in labels.numpy()]

        minority_weight = np.max(weights)
        avg_weight = np.mean(weights)
        
        sample_weights.append(0.6 * minority_weight + 0.4 * avg_weight) 
    
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

def get_cosine_schedule_with_warmup(optimizer, num_warmup_epochs, num_training_epochs, min_lr=1e-6):
    def lr_lambda(current_epoch):
        if current_epoch < num_warmup_epochs:
            return float(current_epoch) / float(max(1, num_warmup_epochs))
        
        progress = float(current_epoch - num_warmup_epochs) / float(max(1, num_training_epochs - num_warmup_epochs))
        cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
        
        base_lr = optimizer.defaults['lr']
        return max(min_lr / base_lr, cosine_decay)
    
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

if __name__ == "__main__":
    # TF32
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    
    # Device setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    print("\n" + "="*60)
    print("Sleep Apnea Detection Training")
    print("="*60)
    print(f"Device: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"CUDA Version: {torch.version.cuda}")
    print(f"Batch Size: {config.BATCH_SIZE}")
    print(f"Gradient Accumulation: {config.GRADIENT_ACCUMULATION_STEPS}")
    print(f"Effective Batch Size: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION_STEPS}")
    print(f"Num Workers: {config.NUM_WORKERS}")
    print(f"Mixed Precision: {config.USE_MIXED_PRECISION}")
    print(f"Cache Size: {config.CACHE_SIZE}")
    print("="*60)
    
    # Create Dataset (now loads from multiple files automatically)
    print(f"\nLoading preprocessed data from {config.PREPROCESSED_DIR}...")
    dataset = SleepApneaDataset(
        preprocessed_dir=config.PREPROCESSED_DIR,
        context_epochs=config.CONTEXT_EPOCHS,
        output_epochs=config.OUTPUT_EPOCHS,
        use_compression=config.USE_COMPRESSION,
        cache_size=config.CACHE_SIZE,
    )

    # Train/Val Split By Folder (Splitting by patient to avoid overlaps)
    all_folder_ids = list(dataset.folder_metadata.keys())
    np.random.seed(42) #Set seed for testing
    np.random.shuffle(all_folder_ids)

    split_idx = int(len(all_folder_ids) * (1 - config.VAL_SPLIT))
    train_folder_ids = all_folder_ids[:split_idx]
    val_folder_ids = all_folder_ids[split_idx:]

    print(f"\nFolder-level split:")
    print(f"    Total folders: {len(all_folder_ids)}")
    print(f"    Training folders: {len(train_folder_ids)} ({train_folder_ids[:5]}...)")
    print(f"    Validation folders: {len(val_folder_ids)} ({val_folder_ids[:5]}...)")

    train_indices, val_indices = dataset._get_folder_split_indices(train_folder_ids, val_folder_ids)

    print(f"\nDataset split:")
    print(f"    Training samples: {len(train_indices)}")
    print(f"    Validation samples: {len(val_indices)}")

    # Dataset Subsets
    train_dataset = Subset(dataset, train_indices)
    val_dataset = Subset(dataset, val_indices)

    # Balanced Sampler
    print("\n" + "="*60)
    print("Creating Balanced Sampler")
    print("="*60)
    train_sampler = create_balanced_sampler(train_dataset, range(len(train_dataset)))
    
    # DataLoaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config.BATCH_SIZE, 
        sampler=train_sampler,
        num_workers=config.NUM_WORKERS,
        pin_memory=True if device.type == 'cuda' else False,
        persistent_workers=(config.NUM_WORKERS > 0),
        prefetch_factor=8 if config.NUM_WORKERS > 0 else None,
        drop_last=True,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=config.NUM_WORKERS,
        pin_memory=True if device.type == 'cuda' else False,
        persistent_workers=(config.NUM_WORKERS > 0),
        prefetch_factor=8 if config.NUM_WORKERS > 0 else None,
    )
    
    # Initialize model
    print("\n" + "="*40)
    print("Initializing model...")
    print("="*40)
    
    model = MultiEpochSleepApneaDetector(
        context_epochs=config.CONTEXT_EPOCHS,
        output_epochs=config.OUTPUT_EPOCHS,
        num_classes=config.NUM_CLASSES,
        dropout=config.DROPOUT,
        lstm_enabled=config.USE_LSTM,
        lstm_layers=config.LSTM_LAYERS,
        transformer_encoder_enabled=config.USE_TRANSFORMER_ENCODER,
        transformer_layers=config.TRANSFORMER_LAYERS,
        use_preextracted_features=config.USE_PREEXTRACTED_FEATURES
    ).to(device)
    
    # Loss (Focal Loss instead of weight cross entropy to target OSA and Hypo)
    criterion = FocalLoss(
        alpha=config.CLASS_WEIGHTS,
        gamma=config.FOCAL_GAMMA,
        label_smoothing=config.LABEL_SMOOTHING,
        minority_gradient_scale=config.MINORITY_GRADIENT_SCALE
    )

    # Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.LEARNING_RATE,
        weight_decay=config.OPTIMIZER_WEIGHT_DECAY,
        fused=True if device.type == 'cuda' else False
    )
    
    # Scheduler
    '''
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=config.SCHEDULER_PATIENCE
    )
    '''
    scheduler = get_cosine_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_epochs=config.LEARNING_RATE_WARMUP_EPOCHS,
        num_training_epochs=config.NUM_EPOCHS,
        min_lr=config.LEARNING_RATE_MIN
    )
    
    # Trainer
    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        output_dir=config.OUTPUT_DIR,
        scheduler=scheduler,
        patience=config.PATIENCE,
        gradient_accumulation_steps=config.GRADIENT_ACCUMULATION_STEPS,
        use_mixed_precision=config.USE_MIXED_PRECISION,
    )
    
    # Start training
    trainer.train(num_epochs=config.NUM_EPOCHS)
    
    print("\nTraining script complete!")


Sleep Apnea Detection Training
Device: cuda
GPU: NVIDIA A100-SXM4-80GB
CUDA Version: 12.4
Batch Size: 64
Gradient Accumulation: 8
Effective Batch Size: 512
Num Workers: 8
Mixed Precision: True
Cache Size: 48

Loading preprocessed data from /home/jwethere/Preprocessed...

Loading dataset index from /home/jwethere/Preprocessed/index.json...

Initialized SleepApneaDataset:
    Total folders: 48
    Total epochs: 138,943
    Valid sequences: 138,031
    Context epochs: 20
    Output epochs: 10
    Cache size: 48 folders

Folder-level split:
    Total folders: 48
    Training folders: 38 (['30', '43', '29', '46', '27']...)
    Validation folders: 10 (['12', '25', '21', '50', '23']...)

Dataset split:
    Training samples: 108592
    Validation samples: 29439

Creating Balanced Sampler

Calculating class distribution for balanced sampling...


Counting classes: 100%|██████████| 108592/108592 [00:06<00:00, 15537.39it/s]



Class distribution:
    No Event: 869,411 (80.1%)
    OSA:      77,524 (7.1%)
    Hypopnea: 138,985 (12.8%)

Sampling weights: [1.17, 6.35, 4.22]

Initializing model...

Initializing MultiEpochSleepApneaDetector: 
    Context Epochs: 20
    Output Epochs: 10
    Number of Classes: 3
    Dropout: 0.4
    Pre-extracted Features: True
    Using LSTM -> Positional Transformer Architecture
    LSTM Layers: 2
    Transformer Layers: 4

Model initialized successfully.
AST Feature Dimension: 768

Initialized FocalLoss:
    Gamma (focusing): 2.0
    Alpha (class weights): tensor([1., 4., 6.])
    Label Smoothing: 0.0
    Minority Gradient Scale: 2.0

Initialized Trainer:
    Device: cuda
    Output Directory: /home/jwethere/Model_Output
    Training Batches: 1696
    Val Batches: 460
    Patience: 10 epochs
    Mixed Precision: True
    Gradient Accumulation: 8

Starting training for 50 epochs...

Epoch 1/50
----------------------------------------
Learning Rate: 0.0000e+00



Epoch 1 Summary:
    Train Loss: 2.9835, Train Acc: 35.05%
    Val Loss: 1.6014, Val Acc: 23.59%
    Val F1 (Macro): 0.2004
    Val F1 (Weighed): 0.1936
    Val F1 (Minority): 0.1327

    Per-Class Metrics:
    Class           Count    Accuracy     F1 Score  
    --------------------------------------------------
    No Event        244842    20.71%      0.3358
    OSA             19130     64.63%      0.1271
    Hypopnea        30418     20.97%      0.1384
    New best model saved (Weighted F1: 0.1936)

Epoch 2/50
----------------------------------------
Learning Rate: 1.6000e-05



Epoch 2 Summary:
    Train Loss: 1.5713, Train Acc: 32.24%
    Val Loss: 1.0189, Val Acc: 33.20%
    Val F1 (Macro): 0.3256
    Val F1 (Weighed): 0.3215
    Val F1 (Minority): 0.2843

    Per-Class Metrics:
    Class           Count    Accuracy     F1 Score  
    --------------------------------------------------
    No Event        244842    25.99%      0.4083
    OSA             19130     72.04%      0.3673
    Hypopnea        30418     66.86%      0.2013
    New best model saved (Weighted F1: 0.3215)

Epoch 3/50
----------------------------------------
Learning Rate: 3.2000e-05


...


Epoch 36 Summary:
    Train Loss: 0.1511, Train Acc: 89.95%
    Val Loss: 3.7549, Val Acc: 78.21%
    Val F1 (Macro): 0.5065
    Val F1 (Weighed): 0.4878
    Val F1 (Minority): 0.3198

    Per-Class Metrics:
    Class           Count    Accuracy     F1 Score  
    --------------------------------------------------
    No Event        244842    87.89%      0.8797
    OSA             19130     46.87%      0.4293
    Hypopnea        30418     19.99%      0.2103
    No improvement for 6 epochs.

Epoch 37/50
----------------------------------------
Learning Rate: 1.7632e-05



Epoch 37 Summary:
    Train Loss: 0.1457, Train Acc: 90.38%
    Val Loss: 3.8478, Val Acc: 77.49%
    Val F1 (Macro): 0.5084
    Val F1 (Weighed): 0.4901
    Val F1 (Minority): 0.3251

    Per-Class Metrics:
    Class           Count    Accuracy     F1 Score  
    --------------------------------------------------
    No Event        244842    86.59%      0.8751
    OSA             19130     50.42%      0.4342
    Hypopnea        30418     21.22%      0.2159
    No improvement for 7 epochs.

Epoch 38/50
----------------------------------------
Learning Rate: 1.5374e-05



Epoch 38 Summary:
    Train Loss: 0.1446, Train Acc: 90.42%
    Val Loss: 3.6613, Val Acc: 77.39%
    Val F1 (Macro): 0.5088
    Val F1 (Weighed): 0.4905
    Val F1 (Minority): 0.3259

    Per-Class Metrics:
    Class           Count    Accuracy     F1 Score  
    --------------------------------------------------
    No Event        244842    86.50%      0.8746
    OSA             19130     47.70%      0.4276
    Hypopnea        30418     22.78%      0.2242
    No improvement for 8 epochs.

Epoch 39/50
----------------------------------------
Learning Rate: 1.3235e-05



Epoch 39 Summary:
    Train Loss: 0.1435, Train Acc: 90.55%
    Val Loss: 3.8603, Val Acc: 78.06%
    Val F1 (Macro): 0.5057
    Val F1 (Weighed): 0.4870
    Val F1 (Minority): 0.3189

    Per-Class Metrics:
    Class           Count    Accuracy     F1 Score  
    --------------------------------------------------
    No Event        244842    87.68%      0.8791
    OSA             19130     45.76%      0.4218
    Hypopnea        30418     20.92%      0.2161
    No improvement for 9 epochs.

Epoch 40/50
----------------------------------------
Learning Rate: 1.1226e-05


Epoch 40 [Training]:  72%|███████▏  | 1214/1696 [00:25<00:09, 49.96it/s, Loss=0.1433, Acc=90.81%, GPU_Mem=0.6GB, GPU%=42%, Temp=35°C, RAM=6.9%]